# Taking the vision encoder out of a multimodal model

A "multimodal" model that reads images is usually two models in a trenchcoat: an image
encoder, and an ordinary text LLM. The encoder turns a picture into a handful of vectors,
and those vectors get spliced into the LLM's input right where an `<image>` placeholder
token sat. The LLM never sees pixels. It just sees vectors that happen to have come from
a picture.

Once you know that, a question suggests itself: if they're two separate things, can we
pull them apart?

Today we take `zai-org/GLM-4.6V-Flash` and keep only the encoder. We'll end up with a
model that takes an image and gives back vectors — no text, no chat, just the visual
front end on its own.

Some things to watch for as we go:

- A model on disk is not a magic binary. It's a dictionary: names to arrays of numbers.
- The names are structured, and that structure is what makes this surgery possible.
- We will check our work twice, in two different ways, before believing anything.

One warning up front, because it shapes the whole exercise. What we build is *not* a
model you can ask questions. It has no way to produce words. It's a component. We'll
come back to why that matters at the end.

## Setting up

Nothing exciting here, but two lines are worth reading.

The dtype choice: modern GPUs have a 16-bit float called `bfloat16` that these models are
trained in. Cards older than NVIDIA's Ampere generation don't have hardware for it, so we
fall back to plain 32-bit floats. Slower and twice the memory, but correct.

The other thing is that this whole notebook fits on a modest GPU. The encoder is about
1.8 GB. That's on purpose, and step 1 explains how we avoid downloading the other 19 GB.

In [1]:
import gc
import json
import shutil
import time
from pathlib import Path

import torch
from safetensors import safe_open
from safetensors.torch import save_file

# safe_open lets us peek inside a .safetensors file without loading it.
# save_file writes a new one. Those two are the whole toolkit for this notebook.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32   # overridden to bfloat16 below if the GPU supports it

if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"{gpu.name}, {gpu.total_memory / 1e9:.1f} GB, compute capability {gpu.major}.{gpu.minor}")
    if gpu.major >= 8:
        DTYPE = torch.bfloat16          # Ampere or newer
    else:
        print("no bfloat16 on this card, using float32 instead")
    if gpu.total_memory / 1e9 < 4:
        print("under 4 GB of VRAM, dropping to CPU")
        DEVICE = "cpu"

print(f"device={DEVICE} dtype={DTYPE}")

REPO = "zai-org/GLM-4.6V-Flash"
SOURCE = Path.home() / "models" / "GLM-4.6V-Flash"        # what we download
OUTPUT = Path.home() / "models" / "GLM-4.6V-Flash-vision"  # what we build

NVIDIA GeForce RTX 2060, 6.4 GB, compute capability 7.5
no bfloat16 on this card, using float32 instead
device=cuda dtype=torch.float32


## Step 1. Get the weights (but not all of them)

Large models are split across several files, called shards, because a single 20 GB file is
awkward to download and awkward to load. A small JSON index says which tensor lives in
which shard.

We can use that index to be lazy. If every tensor we care about happens to sit in one
shard, we only need that shard. Let's look before we download.

In [2]:
from huggingface_hub import hf_hub_download

SOURCE.mkdir(parents=True, exist_ok=True)

# The index is a few kilobytes of JSON. Downloading it first lets us decide
# which multi-gigabyte shards we actually need.
index_path = hf_hub_download(REPO, "model.safetensors.index.json", local_dir=SOURCE)
index = json.loads(Path(index_path).read_text())

# index["weight_map"] maps every tensor name -> the shard file holding it.

# Every vision tensor in this model has a name starting with "model.visual."
# Which shard files do those names point to?
vision_shards = sorted({
    shard for name, shard in index["weight_map"].items()
    if name.startswith("model.visual.")
})

print("shards containing vision tensors:", vision_shards)
print("total shards in the model       :", len(set(index["weight_map"].values())))

/home/user/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shards containing vision tensors: ['model-00004-of-00004.safetensors']
total shards in the model       : 4


One shard out of four. That is a lucky break and it means a 4.7 GB download instead of
20.6 GB.

It is luck, though, not a guarantee. If the authors re-upload the model with a different
split, this could become two shards or three. So rather than hard-coding "download file
number four", we assert what we expect. If the assumption ever stops holding, the notebook
stops with a clear message instead of quietly doing the wrong thing.

Get into the habit. An assert costs one line and saves an afternoon.

In [3]:
assert len(vision_shards) == 1, (
    f"expected all vision tensors in one shard, found {len(vision_shards)}. "
    "The download step below needs updating."
)

wanted = [
    "config.json",                    # architecture description
    "preprocessor_config.json",       # how to turn a JPEG into model input
    "video_preprocessor_config.json",
    vision_shards[0],                 # the weights themselves
]

start = time.time()
for filename in wanted:
    try:
        hf_hub_download(REPO, filename, local_dir=SOURCE)
        print("got", filename)
    except Exception as err:
        print("skipped", filename, f"({type(err).__name__})")
print(f"{time.time() - start:.0f}s")

got config.json
got preprocessor_config.json


got video_preprocessor_config.json


got model-00004-of-00004.safetensors
27s


### A look at the config

`config.json` is the model's blueprint: how many layers, how wide, what activation
function. It has a `vision_config` section describing only the encoder, which is the part
we want.

A few of these numbers show up later, so it's worth a glance now rather than being
surprised by them in step 6.

In [4]:
config = json.loads((SOURCE / "config.json").read_text())
vision_config = config["vision_config"]

for key in ["depth", "hidden_size", "num_heads", "patch_size", "image_size",
            "spatial_merge_size", "out_hidden_size"]:
    print(f"{key:20} {vision_config[key]}")

depth                24
hidden_size          1536
num_heads            12
patch_size           14
image_size           336
spatial_merge_size   2
out_hidden_size      4096


Reading that back in English: the encoder is a 24-layer transformer, 1536 units wide, with
12 attention heads. It chops the image into 14x14 pixel patches. At the end it projects
everything up to 4096 dimensions.

Why 4096? Because that's the width of the language model this encoder was built to feed.
The final projection exists purely to make the encoder's output fit the LLM's input. That
single number is the seam between the two halves of the model, and it's the reason this
whole exercise works.

## Step 2. Look inside the file

Here's the part that surprises people the first time.

A `.safetensors` file is a dictionary. Keys are strings like
`model.visual.blocks.7.attn.qkv.weight`, values are arrays of numbers. That's the entire
format. No code, no graph, nothing clever.

Read the key above from right to left: it's the weight of the qkv projection, in the
attention block, of layer 7, of the visual tower, of the model. The dots are a path
through the model's structure, and the library reconstructs the object from those paths
when it loads.

So let's open the shard and see what's actually in there.

In [5]:
shard_path = SOURCE / vision_shards[0]

# safe_open reads the file's header without loading any weights into memory,
# so this is instant even on a 4.7 GB file.
shapes = {}
with safe_open(shard_path, framework="pt") as f:
    for name in f.keys():
        # get_slice gives a lazy handle; asking only for its shape reads no data
        shapes[name] = tuple(f.get_slice(name).get_shape())

# Sorting by name happens to group layers together, which makes printing readable.
vision_names = sorted(n for n in shapes if n.startswith("model.visual."))
other_names = sorted(n for n in shapes if not n.startswith("model.visual."))

print(f"tensors in this shard: {len(shapes)}")
print(f"  vision             : {len(vision_names)}")
print(f"  language model     : {len(other_names)}")
print()
for name in vision_names[:5]:
    print(f"  {name:<52} {shapes[name]}")
print("  ...")

tensors in this shard: 235
  vision             : 181
  language model     : 54

  model.visual.blocks.0.attn.proj.weight               (1536, 1536)
  model.visual.blocks.0.attn.qkv.weight                (4608, 1536)
  model.visual.blocks.0.mlp.down_proj.weight           (1536, 4096)
  model.visual.blocks.0.mlp.gate_proj.weight           (4096, 1536)
  model.visual.blocks.0.mlp.up_proj.weight             (4096, 1536)
  ...


Notice the shard holds a mix. Vision tensors and language tensors sit side by side in the
same file, because the split into shards is just "fill a file until it's about 5 GB, start
a new one". It has nothing to do with the model's structure.

Now let's count parameters. A parameter is a single number, and a tensor's parameter count
is the product of its shape. So a `[1536, 1536]` matrix holds 1536 x 1536 = 2,359,296 of
them.

In [6]:
def count_params(shape):
    # A tensor holds the product of its dimensions: [1536, 1536] is 2,359,296 numbers.
    total = 1
    for dim in shape:
        total *= dim
    return total

vision_params = sum(count_params(shapes[n]) for n in vision_names)

print(f"vision parameters: {vision_params:,}")
# Two bytes per number in bfloat16, four in float32. Precision costs disk.
print(f"as bfloat16      : {vision_params * 2 / 1e9:.2f} GB")
print(f"as float32       : {vision_params * 4 / 1e9:.2f} GB")

vision parameters: 892,498,432
as bfloat16      : 1.78 GB
as float32       : 3.57 GB


892 million. For comparison the full model is 10.3 billion, so the encoder is about 8.7%
of the whole thing. The other 91% is the language model we're throwing away.

Let's also see where those 892 million sit. Grouping by submodule tells you what a vision
transformer is actually made of.

In [7]:
import re
from collections import Counter

groups = Counter()
for name in vision_names:
    short = name[len("model.visual."):]
    # Collapse blocks.0..., blocks.1... into one row so we see the stack as a whole
    short = re.sub(r"blocks\.\d+\..*", "blocks.*  (the 24 transformer layers)", short)
    # ...and merge each module's weight and bias into a single line
    short = re.sub(r"\.(weight|bias)$", "", short)
    groups[short] += count_params(shapes[name])

for group, count in groups.most_common():
    share = 100 * count / vision_params
    print(f"  {group:<42} {count:>12,}  {share:5.1f}%")

  blocks.*  (the 24 transformer layers)       679,550,976   76.1%
  merger.down_proj                             56,098,816    6.3%
  merger.gate_proj                             56,098,816    6.3%
  merger.up_proj                               56,098,816    6.3%
  downsample                                   25,169,920    2.8%
  merger.proj                                  16,777,216    1.9%
  patch_embed.proj                              1,807,872    0.2%
  embeddings.position_embedding                   884,736    0.1%
  merger.post_projection_norm                       8,192    0.0%
  post_conv_layernorm                               1,536    0.0%
  post_layernorm                                    1,536    0.0%


Three quarters of the encoder is the stack of transformer layers, which is what you'd
expect. The `merger` is the projection up to 4096 that we talked about — it's small in
tensor count but not in parameters, because 4096-wide matrices add up fast.

## Step 3. The surgery

We now do the actual extraction, and it's about six lines.

The trick is in the names. Inside the full model, the encoder lives under
`model.visual.something`. Transformers also ships a standalone class for just this encoder,
called `Glm4vVisionModel`, and that class expects its weights to be named plain
`something` — no prefix, because from its point of view it *is* the whole model.

So the surgery is: keep the tensors whose names start with `model.visual.`, and chop that
prefix off. That's it. Same numbers, different names.

We read one tensor at a time rather than loading the shard in one go. On a machine with
limited RAM that's the difference between working and swapping.

In [8]:
OUTPUT.mkdir(parents=True, exist_ok=True)

PREFIX = "model.visual."
weights = {}

# This loop is the entire operation. One tensor at a time, so peak memory stays
# at the size of the largest tensor rather than the size of the file.
with safe_open(shard_path, framework="pt") as f:
    for name in vision_names:
        new_name = name[len(PREFIX):]      # "model.visual.blocks.0..." -> "blocks.0..."
        # get_tensor copies the numbers out unchanged. No maths, no conversion.
        weights[new_name] = f.get_tensor(name)

save_file(weights, OUTPUT / "model.safetensors", metadata={"format": "pt"})

print(f"saved {len(weights)} tensors")
print("example rename:")
print(f"  {vision_names[0]}")
print(f"  -> {vision_names[0][len(PREFIX):]}")

saved 181 tensors
example rename:
  model.visual.blocks.0.attn.proj.weight
  -> blocks.0.attn.proj.weight


Did we produce the structure the class expects? We can sanity-check by looking at the
top-level names. Each one should correspond to an attribute on `Glm4vVisionModel`.

In [9]:
top_level = sorted({name.split(".")[0] for name in weights})
print("top-level modules in our file:")
for name in top_level:
    print("  ", name)

expected = {"blocks", "downsample", "embeddings", "merger",
            "patch_embed", "post_conv_layernorm", "post_layernorm"}
assert set(top_level) == expected, f"unexpected: {set(top_level) ^ expected}"
print("\nmatches what Glm4vVisionModel defines")

top-level modules in our file:
   blocks
   downsample
   embeddings
   merger
   patch_embed
   post_conv_layernorm
   post_layernorm

matches what Glm4vVisionModel defines


## Step 4. Write a config for it

Weights alone aren't enough. Loading a model means building the empty structure first and
then pouring the numbers in, and the structure comes from `config.json`.

The original config describes the whole multimodal model. We want a config that describes
only the encoder. Rather than editing JSON by hand, we hand the vision section to the
`Glm4vVisionConfig` class and let it produce the file. If a field name changes in a future
version of transformers, this keeps working; hand-edited JSON wouldn't.

One field the class does *not* fill in for you is `architectures`. That's the hint
`from_pretrained` uses to pick which Python class to build. Leave it out and loading fails
with a confusing error, so we set it ourselves. This catches people out regularly.

In [10]:
from transformers import Glm4vVisionConfig

fields = {k: v for k, v in vision_config.items() if k != "model_type"}
config_out = Glm4vVisionConfig(**fields)
config_out.architectures = ["Glm4vVisionModel"]   # easy to forget, breaks loading
config_out.dtype = str(DTYPE).replace("torch.", "")
config_out.save_pretrained(OUTPUT)

# The image preprocessor travels with the model: it knows the resize rules and the
# normalisation constants. Without it you'd have to guess how to prepare a JPEG.
for filename in ["preprocessor_config.json", "video_preprocessor_config.json"]:
    if (SOURCE / filename).exists():
        shutil.copy(SOURCE / filename, OUTPUT / filename)

print("wrote:")
for path in sorted(OUTPUT.iterdir()):
    print(f"  {path.name:<36} {path.stat().st_size / 1e6:8.1f} MB")

wrote:
  config.json                               0.0 MB
  model.safetensors                      1785.0 MB
  preprocessor_config.json                  0.0 MB
  video_preprocessor_config.json            0.0 MB


Notice there's no tokenizer in that list. A tokenizer converts text to numbers, and this
model has no text path at all, so shipping one would be misleading.

## Step 5. Check our work

This is the step students skip and researchers regret skipping.

We're going to verify twice, because the two checks catch different failures.

**Check one: are the numbers the same?** We only copied tensors, so every value in our file
should be bit-for-bit identical to the original. If not, something corrupted them.

**Check two: does the model behave the same?** Even with perfect weights, we could have
written a wrong config, or mismatched a name, and ended up with a model that loads but
computes something different. So we build the model two ways and compare the outputs.

Neither check alone is enough. The first says the ingredients are right; the second says we
assembled them right.

In [11]:
mismatches = []

with safe_open(shard_path, framework="pt") as f:
    for name in vision_names:
        original = f.get_tensor(name)
        ours = weights[name[len(PREFIX):]]
        # torch.equal is exact: same dtype, same shape, same bits. Not "close to".
        if not torch.equal(original, ours):
            mismatches.append(name)

print(f"compared {len(vision_names)} tensors")
print("mismatches:", len(mismatches) if mismatches else "none, all identical")
assert not mismatches

compared 181 tensors
mismatches: none, all identical


Now the behavioural check.

We build two copies of the encoder. Copy A is loaded by hand, straight from the original
shard. Copy B is loaded normally with `from_pretrained` from the checkpoint we just wrote.
Then we push the same input through both.

If they disagree, the problem is in our config or our naming, because we already proved the
weights are identical.

One detail about the input: this encoder does not take an image tensor shaped like a
picture. The preprocessor has already flattened the image into a list of patches by the time
the model sees it, so we build a fake batch of patches directly. `grid_thw` tells the model
how those patches were laid out — time, height, width — since a flat list has lost that.

In [12]:
from transformers import Glm4vVisionModel

torch.manual_seed(0)

# Build an input by hand. The encoder expects already-flattened patches, so we
# make a grid of them rather than an image. Size is arbitrary; 16x16 is quick.
merge = config_out.spatial_merge_size
grid_h = grid_w = 8 * merge
# Each patch is a flat vector: 3 colour channels x 2 time steps x 14 x 14 pixels
patch_values = config_out.in_channels * config_out.temporal_patch_size * config_out.patch_size ** 2

fake_patches = torch.randn(grid_h * grid_w, patch_values)
grid = torch.tensor([[1, grid_h, grid_w]])     # one image, grid_h by grid_w patches

def run(model):
    # .eval() turns off dropout and similar training-only behaviour, so repeated
    # runs give identical numbers. Without it this comparison would be noise.
    model = model.to(DEVICE, DTYPE).eval()
    with torch.no_grad():
        out = model(fake_patches.to(DEVICE, DTYPE), grid_thw=grid.to(DEVICE))
    result = out.last_hidden_state.float().cpu()
    # Free the GPU before the caller loads the second model.
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result

# A: built by hand from the original file
model_a = Glm4vVisionModel(config_out)
with safe_open(shard_path, framework="pt") as f:
    model_a.load_state_dict({n[len(PREFIX):]: f.get_tensor(n) for n in vision_names})
output_a = run(model_a)

# B: loaded the normal way from our new checkpoint
output_b = run(Glm4vVisionModel.from_pretrained(OUTPUT, dtype=DTYPE, low_cpu_mem_usage=True))

difference = (output_a - output_b).abs().max().item()
print(f"output shape        {tuple(output_a.shape)}")
print(f"largest difference  {difference}")
print("identical" if difference == 0.0 else "SOMETHING IS WRONG, do not use this checkpoint")

Loading weights:   0%|          | 0/181 [00:00<?, ?it/s]

Loading weights:  12%|█▏        | 22/181 [00:00<00:00, 204.04it/s]

Loading weights:  28%|██▊       | 51/181 [00:00<00:00, 236.25it/s]

Loading weights:  41%|████▏     | 75/181 [00:00<00:00, 181.58it/s]

Loading weights:  52%|█████▏    | 95/181 [00:00<00:00, 130.64it/s]

Loading weights:  61%|██████    | 110/181 [00:00<00:00, 118.84it/s]

... (5 outputs left out)

Loading weights:  94%|█████████▍| 171/181 [00:02<00:00, 33.84it/s]

Loading weights:  98%|█████████▊| 177/181 [00:02<00:00, 26.16it/s]

Loading weights: 100%|██████████| 181/181 [00:02<00:00, 66.29it/s]

output shape        (64, 4096)
largest difference  0.0
identical


Zero. Not "small", not "close enough" — exactly zero, because we did no arithmetic on the
weights, only moved them.

It's worth being precise about what this proves and what it doesn't. We compared our
checkpoint against the same weights loaded a different way. We did *not* compare against
the full multimodal model running end to end, which would need about 21 GB of memory and
won't fit here. So this confirms our plumbing, not the encoder's behaviour inside the
original system. Given the bit-identical check above, that's a reasonable place to stop.

Knowing the difference between "I verified X" and "I verified something that implies X" is
most of what makes a result trustworthy.

## Step 6. Run it on a picture

Enough checking. Let's put an image through.

Point `IMAGE_PATH` at any file you like. If you leave it as `None` we generate a simple
pattern so the notebook runs anywhere.

In [13]:
from PIL import Image, ImageDraw
from transformers import AutoImageProcessor

IMAGE_PATH = None      # e.g. Path("cat.jpg")

if IMAGE_PATH is None:
    image = Image.new("RGB", (448, 448), (18, 22, 30))
    draw = ImageDraw.Draw(image)
    for offset in range(0, 224, 16):
        draw.rectangle([offset, offset, 447 - offset, 447 - offset],
                       outline=(255 - offset, (offset * 3) % 256, 128))
else:
    image = Image.open(IMAGE_PATH).convert("RGB")

# The processor does the resizing, normalising and patch-splitting. It came from
# preprocessor_config.json, which is why we copied that file across in step 4.
processor = AutoImageProcessor.from_pretrained(OUTPUT)
inputs = processor(images=image, return_tensors="pt")

print("what the processor produced:")
print(f"  pixel_values   {tuple(inputs['pixel_values'].shape)}")
print(f"  image_grid_thw {inputs['image_grid_thw'].tolist()}")

what the processor produced:
  pixel_values   (1024, 1176)
  image_grid_thw [[1, 32, 32]]


The 448x448 image became 1024 rows. That's 448/14 = 32 patches across and 32 down, and
32 x 32 = 1024. Each row is one flattened patch. This is what I meant earlier about the
model never seeing a picture — by this point it's a list.

In [14]:
model = Glm4vVisionModel.from_pretrained(OUTPUT, dtype=DTYPE, low_cpu_mem_usage=True)
model = model.to(DEVICE).eval()

with torch.no_grad():
    output = model(inputs["pixel_values"].to(DEVICE, DTYPE),
                   grid_thw=inputs["image_grid_thw"].to(DEVICE))

# Back to float32 on the CPU so the statistics below print sensibly.
features = output.last_hidden_state.float().cpu()

print(f"features {tuple(features.shape)}")
print(f"  mean {features.mean():+.3f}   std {features.std():.3f}")
print(f"  min  {features.min():+.3f}   max {features.max():+.3f}")

Loading weights:   0%|          | 0/181 [00:00<?, ?it/s]

Loading weights:  13%|█▎        | 23/181 [00:00<00:00, 224.75it/s]

Loading weights:  25%|██▌       | 46/181 [00:00<00:01, 129.04it/s]

Loading weights:  34%|███▍      | 62/181 [00:00<00:01, 116.64it/s]

Loading weights:  41%|████▏     | 75/181 [00:00<00:01, 90.71it/s] 

Loading weights:  48%|████▊     | 86/181 [00:00<00:01, 86.91it/s]

... (6 outputs left out)

Loading weights:  90%|█████████ | 163/181 [00:01<00:00, 86.45it/s]

Loading weights:  95%|█████████▌| 172/181 [00:02<00:00, 34.43it/s]

Loading weights: 100%|██████████| 181/181 [00:02<00:00, 73.26it/s]

features (256, 4096)
  mean +0.007   std 1.005
  min  -9.747   max +9.626


1024 patches went in, 256 vectors came out. The encoder merges patches in 2x2 groups
(`spatial_merge_size` is 2), so four patches become one output vector: 1024 / 4 = 256.

Those 256 vectors, each 4096 numbers long, are the finished product. In the original model
they'd be spliced into the language model's input, and the LLM would carry on as though
someone had typed 256 tokens describing the picture.

## What we ended up with

A 1.8 GB file that turns images into vectors. Roughly a tenth of the model we started with.

Now the honest part, and the thing I most want you to take away.

Earlier in the course we did the opposite operation: we deleted the encoder and kept the
language model. That produced something you can actually use — you load it, you type a
question, it answers. It's a finished model.

This is not that. There's no head on it. It can't classify, caption, or answer anything. It
turns pictures into numbers and stops. To do something with it you'd have to attach
something else: a classifier, or an adapter feeding a different LLM.

The operations look symmetric and aren't. A decoder with a language head is complete by
itself. An encoder is a component that produces representations for something downstream.
Cutting a model in half gives you one piece that works and one piece that needs a partner,
and which is which depends on where the head lives.

### Things to try

1. Change `IMAGE_PATH` to a real photo. Do the feature statistics look different from the
   synthetic pattern? Would you expect them to?
2. Feed it two different images and compare the mean feature vectors with cosine
   similarity. Do similar pictures land closer together?
3. Try a 224x224 image. Predict the number of output vectors before you run it.
4. Delete `config_out.architectures = [...]` and reload. Read the error. That's the error
   you'll get in six months when you forget this line.